# 10 · SVM pro diagnostiku čerpadel

Praktická část cvičení 10. Na provozních záznamech odstředivých čerpadel porovnáte lineární model se SVM s jádry,
**samostatně vyladíte C a γ** křížovou validací, zvolíte rozhodovací práh podle ceny chyb a výsledek jednou otestujete.
Potom vyzkoušíte dvě další úlohy SVM z přednášky: jednotřídní SVM naučenou bez označených poruch a regresi charakteristiky
čerpadla s ε-trubicí včetně extrapolace. Každé nastavení, které zde ladíte, jste viděli v edukační části `svm-visual-cz.ipynb`.

Krátké spustitelné ukázky používají malá ilustrační data a proměnné s předponou `demo_`. Spouštějte je v pořadí;
ukazují syntaxi celého postupu. Vlastní řešení pište do prázdných kódových buněk pod úkoly.
Testovací data se otevřou až v kroku 7. Po dokončení spusťte `Restart Kernel and Run All` a ověřte, že řešení běží v pořadí.

## Příprava prostředí a načtení dat

1. Otevřete notebook v Jupyteru (balíčky popisuje README) nebo v Google Colab.
2. Spusťte ukázku níže. Importuje všechny potřebné nástroje a připraví malá ilustrační data `demo_`.
3. Připravená buňka pod ní načte data do tabulky `raw` přímo z repozitáře předmětu na GitHubu
   (`10-SVM/data/pumps.csv`) a nastaví `SEED = 42`. Potřebuje připojení k internetu.

### Ukázka: importy a ilustrační data

In [ ]:
from IPython.display import display                          # Zobrazení tabulek.
import matplotlib.pyplot as plt                              # Grafy.
import numpy as np                                           # Numerické výpočty.
import pandas as pd                                          # Data a tabulky výsledků.
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, cross_val_predict, GridSearchCV  # Dělení, křížová validace a mřížkové hledání.
from sklearn.pipeline import Pipeline                        # Řetězení škálování a modelu.
from sklearn.preprocessing import StandardScaler             # Standardizace.
from sklearn.linear_model import LogisticRegression          # Logistický neuron z přednášky 09.
from sklearn.svm import SVC, SVR, OneClassSVM                # SVM pro klasifikaci, regresi a jednu třídu.
from sklearn.metrics import confusion_matrix, roc_auc_score, mean_absolute_error  # Matice záměn, ROC AUC, MAE.

demo_rng = np.random.default_rng(7)                          # Ilustrační data, ne data úlohy.
demo_X = pd.DataFrame({"a": demo_rng.normal(0, 1, 400), "b": demo_rng.normal(0, 10, 400)})  # Dva příznaky v různých měřítkách.
demo_y = ((demo_X["a"] ** 2 + (demo_X["b"] / 10) ** 2) > 1.6).astype(int)  # Třída 1 vně kružnice: žádná přímka ji neoddělí.
demo_X_dev, demo_X_test, demo_y_dev, demo_y_test = train_test_split(demo_X, demo_y, test_size=0.25, stratify=demo_y, random_state=7)  # Čtvrtina pro test.
demo_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)  # Pět částí se stejným podílem tříd.
print(len(demo_X_dev), len(demo_X_test), "podíl třídy 1:", round(demo_y.mean(), 3))  # Velikosti a podíl.

**Připravený kód:**

In [ ]:
SEED = 42                                                    # Stejné náhodné dělení pro všechny.
DATA_URL = ("https://raw.githubusercontent.com/CVUT-FS-12110/Artificial-Intelligence-and-Neural-Networks/"
            "master/10-SVM/data/pumps.csv")                  # Soubor dat v repozitáři předmětu na GitHubu.
raw = pd.read_csv(DATA_URL)                                  # Stáhnout a načíst tabulku.
display(raw.head())                                          # Prvních pět řádků.

## Úloha a data

Každý řádek je jeden **provozní záznam odstředivého čerpadla** (všechna čerpadla jsou stejného typu) s výsledkem prohlídky `fault`
(1 = nalezena porucha). Data jsou syntetická; jejich popis je v `data/README.md`.

| Sloupec | Význam | Typická hodnota |
|---|---|---|
| `flow` | průtok, pracovní bod | 20 až 80 m³/h |
| `pressure` | výtlačný tlak | 2 až 6 bar |
| `bearing_temp` | teplota ložiska | 60 °C |
| `current` | proud motoru | 28 A |
| `vibration` | rychlost vibrací | 0,6 mm/s |
| `ambient_temp` | teplota okolí | 22 °C |

**Cíl:** označit záznamy, u kterých se má provést prohlídka. **Ceny:** přehlédnutá porucha (FN) stojí **10** jednotek (neplánovaná odstávka),
zbytečná prohlídka (FP) stojí **1** jednotku. Kvalita rozhodnutí se měří **celkovými náklady** 10·FN + 1·FP.
(Náklady se zde neoznačují C, aby se nepletly s parametrem C metody SVM.)

## 1. Prohlídka dat (připraveno)

**Výstup:** podíl poruch, tabulka rozsahů příznaků a dva bodové grafy. Všimněte si, kde leží poruchy vzhledem
k mraku běžných záznamů v grafu průtoku a tlaku.

**Připravený kód:**

In [ ]:
print("řádků:", len(raw), "| podíl poruch:", round(raw["fault"].mean(), 3))  # Velikost a podíl tříd.
display(raw.drop(columns=["record_id", "fault"]).describe().T[["mean", "std", "min", "max"]].round(2))  # Rozsahy vstupů.
fig, axs = plt.subplots(1, 2, figsize=(11, 4))               # Dva grafy vedle sebe.
for ax, col in zip(axs, ["pressure", "bearing_temp"]):       # Tlak a teplota ložiska proti průtoku.
    for value, colour, label in ((0, "#0065BD", "v pořádku"), (1, "#C2410C", "porucha")):
        part = raw[raw["fault"] == value]
        ax.scatter(part["flow"], part[col], s=12, color=colour, label=label, alpha=0.7)
    ax.set(xlabel="flow", ylabel=col)
axs[0].legend()
plt.show()                                                   # Zobrazit grafy.

## 2. Příznaky a pevné rozdělení (připraveno)

**Výstup:** `X` (šest vstupů) a `y` (`fault`); vývojová část `X_dev`, `y_dev` (675 záznamů) a testovací část `X_test`, `y_test`
(225 záznamů), která zůstává zavřená až do kroku 7; rozdělení `CV` (pětinásobná stratifikovaná křížová validace) pro všechna srovnání.

**Připravený kód:**

In [ ]:
FEATURES = ["flow", "pressure", "bearing_temp", "current", "vibration", "ambient_temp"]  # Šest vstupů.
X = raw[FEATURES]                                            # Vstupy; record_id není vstup.
y = raw["fault"]                                             # Cílová třída.
X_dev, X_test, y_dev, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)  # Čtvrtina pro test, stratifikovaně.
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)  # Stejných pět částí pro každý model.
print(f"dev {len(y_dev)} záznamů, {int(y_dev.sum())} poruch | test {len(y_test)} záznamů, {int(y_test.sum())} poruch")

## 3. Funkce pro vyhodnocení a referenční model (připraveno)

**Výstup:** `cv_auc(model)` vrací průměrné ROC AUC pětinásobné křížové validace na vývojových datech
(ROC AUC nezávisí na prahu, proto se používá pro ladění); `evaluate(y_true, y_pred)` vrací počty z matice záměn,
přesnost, úplnost a celkové náklady. Referencí je logistický neuron ze cvičení 09 se škálováním.

**Připravený kód:**

In [ ]:
def cv_auc(model, X_=None, y_=None, cv=None):
    # Průměrné ROC AUC přes části; výchozí jsou vývojová data a CV.
    X_ = X_dev if X_ is None else X_
    y_ = y_dev if y_ is None else y_
    return cross_val_score(model, X_, y_, cv=CV if cv is None else cv, scoring="roc_auc").mean()


def evaluate(y_true, y_pred):
    # Počty, dvě metriky a náklady 10·FN + 1·FP.
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"TN": tn, "FP": fp, "FN": fn, "TP": tp, "přesnost": tp / max(tp + fp, 1),
            "úplnost": tp / max(tp + fn, 1), "náklady": 10 * fn + fp}


reference = Pipeline([("scale", StandardScaler()), ("model", LogisticRegression())])  # Škálování, potom logistický neuron.
print("logistický neuron, CV ROC AUC:", round(cv_auc(reference), 3))  # Kolem 0,5 = ne lépe než náhoda.
print("náklady „vše v pořádku“ na dev:", 10 * int(y_dev.sum()))  # Každá porucha přehlédnuta.

## 4. Lineární model a jádra

**Cíl / výstup:** tabulka `kernel_results` s CV ROC AUC čtyř modelů na vývojových datech:

1. lineární SVM: `Pipeline` s kroky `scale`: `StandardScaler()` → `model`: `SVC(kernel="linear")`;
2. polynomiální jádro stupně 2: `SVC(kernel="poly", degree=2, coef0=1)` po škálování;
3. RBF jádro s výchozím nastavením: `SVC()` po škálování;
4. stejné RBF jádro **bez** škálování.

**Použijte:** `Pipeline`, `StandardScaler`, `SVC`, `cv_auc`, `pd.Series` nebo `pd.DataFrame`.

<details><summary>Nápověda</summary>

Jádro se zadává argumentem `kernel` (`"linear"`, `"poly"`, `"rbf"`; výchozí je `"rbf"` s `C=1` a `gamma="scale"`).
`coef0=1` přidá do polynomiálního jádra (γ·x·z + 1)^d konstantu r = 1, takže obsahuje všechny součiny až do stupně d.
Výsledky sbírejte do slovníku `{název: cv_auc(model)}` a převeďte na `pd.Series(...)`.

</details>

**Zamyslete se:** Proč není lineární model lepší než náhoda (podívejte se na graf v kroku 1)? Proč už pomůže stupeň 2?
Jak moc záleží na škálování a proč (edukační část, oddíl 5)?

### Ukázka: jádra na ilustračních datech

In [ ]:
demo_models = {                                              # Tři jádra po škálování, jedno bez něj.
    "lineární": Pipeline([("scale", StandardScaler()), ("model", SVC(kernel="linear"))]),
    "poly 2": Pipeline([("scale", StandardScaler()), ("model", SVC(kernel="poly", degree=2, coef0=1))]),
    "RBF": Pipeline([("scale", StandardScaler()), ("model", SVC())]),
    "RBF bez škálování": SVC(),
}
demo_results = pd.Series({name: cv_auc(model, demo_X_dev, demo_y_dev, demo_cv) for name, model in demo_models.items()}, name="CV ROC AUC")  # Jedno číslo na model.
display(demo_results.round(3))                               # Kružnice potřebuje jádro a škálování.

**Vlastní kód:**

## 5. Ladění C a γ

**Cíl / výstup:** naučený objekt `GridSearchCV` `grid` pro RBF pipeline, teplotní mapa jeho CV ROC AUC a výsledný
`best_svm` (jeho `best_estimator_`) s počtem podpůrných vektorů.

1. Pipeline `scale` → `model`: `SVC()`; mřížka `model__C` ∈ {0,1; 1; 10; 100}, `model__gamma` ∈ {0,01; 0,1; 1; 10};
   `cv=CV`, `scoring="roc_auc"`. Naučte ji na vývojových datech.
2. Zobrazte průměrné CV skóre jako tabulku C × γ (`pivot_table`) a jako teplotní mapu.
3. Pokud nejlepší hodnota leží **na kraji mřížky**, rozšiřte mřížku tím směrem (a kolem nejlepšího γ) a hledejte znovu.
   Opakujte, dokud se nejlepší CV skóre znatelně zlepšuje (o víc než asi 0,005). Výsledné hledání uložte do `grid`
   a jeho nejlepší model do `best_svm`.
4. Vypište počet podpůrných vektorů `best_svm` (`best_svm[-1].n_support_`) vzhledem k počtu učebních záznamů.

**Použijte:** `GridSearchCV`, `.cv_results_`, `pd.DataFrame(...).pivot_table(...)`, `plt.imshow`, `.best_params_`, `.best_score_`, `.best_estimator_`.

<details><summary>Nápověda</summary>

Parametry kroku pipeline se jmenují `krok__parametr`, zde `model__C` a `model__gamma`. Tabulka:
`pd.DataFrame(grid.cv_results_).pivot_table(index="param_model__C", columns="param_model__gamma", values="mean_test_score")`.
`GridSearchCV` nejlepší kombinaci znovu naučí na všech vývojových datech (výchozí `refit=True`).

</details>

**Zamyslete se:** Kterým směrem jste mřížku rozšířili? Změnilo se C i γ? Jak nejlepší model souvisí se snímkem přednášky
o společné volbě C a γ?

### Ukázka: mřížkové hledání na ilustračních datech

In [ ]:
demo_pipe = Pipeline([("scale", StandardScaler()), ("model", SVC())])  # Škálování a RBF SVM.
demo_grid = GridSearchCV(demo_pipe, {"model__C": [0.1, 1, 10], "model__gamma": [0.1, 1, 10]},  # Devět kombinací.
                         cv=demo_cv, scoring="roc_auc").fit(demo_X_dev, demo_y_dev)  # Každá s křížovou validací.
demo_table = pd.DataFrame(demo_grid.cv_results_).pivot_table(index="param_model__C", columns="param_model__gamma", values="mean_test_score")  # Tabulka C × γ.
display(demo_table.round(3))
plt.imshow(demo_table.values, cmap="Blues", origin="lower")  # Teplotní mapa; řádky C, sloupce γ.
plt.xticks(range(3), demo_table.columns); plt.yticks(range(3), demo_table.index)
plt.xlabel("γ"); plt.ylabel("C"); plt.colorbar(); plt.show()
print(demo_grid.best_params_, round(demo_grid.best_score_, 3), "podpůrné vektory:", demo_grid.best_estimator_[-1].n_support_.sum())

**Vlastní kód:**

## 6. Práh podle ceny chyb

**Cíl / výstup:** skóre `dev_scores` nejlepšího nastavení z křížové validace na vývojových datech, graf celkových nákladů
v závislosti na prahu, práh `best_threshold` s nejnižšími náklady a srovnání s prahem 0 (výchozí práh `predict`).

1. `dev_scores = cross_val_predict(best_svm, X_dev, y_dev, cv=CV, method="decision_function")` – každý záznam dostane skóre od modelu,
   který ho neviděl.
2. Pro prahy od −2 do 1 (krok 0,05) spočtěte náklady `(dev_scores >= práh)`; vykreslete je a najděte minimum.
3. Vypište `evaluate` pro nejlepší práh a pro 0.

**Použijte:** `cross_val_predict`, `np.arange`, `evaluate`, `plt.plot`.

<details><summary>Nápověda</summary>

Skóre SVM je hodnota w·φ(x) + b podobná vzdálenosti, ne pravděpodobnost; `predict` používá práh 0. Záporný práh
označí za poruchu více záznamů: méně FN, více FP. Při poměru cen 10 : 1 bývá nejlepší práh pod nulou.

</details>

**Zamyslete se:** Kolik přehlédnutých poruch a kolik zbytečných prohlídek dávají oba prahy? Proč se práh volí na vývojových
datech, a ne na testovacích?

**Vlastní kód:**

## 7. Závěrečný test (jednou)

**Cíl / výstup:** `test_scores` = `best_svm.decision_function(X_test)`, ROC AUC na testovacích datech a tabulka `test_result`
s `evaluate` pro `best_threshold` a pro 0 a náklady „vše v pořádku“ na testovacích datech. Nic se už neladí.

**Vlastní kód:**

## 8. Jednotřídní SVM bez označených poruch (rozšiřující)

Kroky 8 a 9 jsou rozšiřující: v hodině, pokud zbude čas, jinak doma.

V praxi často chybí označené poruchy; jsou jen záznamy běžného provozu. **Cíl / výstup:** tabulka `oc_results`
s validačním ROC AUC jednotřídní SVM pro ν ∈ {0,02; 0,05; 0,1} a γ ∈ {0,03; 0,1; 0,3; 1}, zvolené nastavení
a ROC AUC výsledného jednotřídního modelu `oc_model` na testu.

1. Rozdělte vývojová data ještě jednou: `X_oc_train, X_oc_val, y_oc_train, y_oc_val` (30 % validace, stratifikovaně, `SEED`).
2. Pro každou kombinaci naučte `Pipeline([("scale", StandardScaler()), ("model", OneClassSVM(nu=..., gamma=...))])`
   **jen na záznamech v pořádku** `X_oc_train[y_oc_train == 0]`; ROC AUC vyhodnoťte na `X_oc_val` se skóre
   `-model.decision_function(...)` (vyšší = neobvyklejší).
3. Nejlepší kombinaci naučte znovu na všech záznamech v pořádku z vývojových dat; vypište ROC AUC na testu, podíl testovacích záznamů
   v pořádku označených jako odchylka a podíl nalezených testovacích poruch (`predict == -1`).

**Použijte:** `train_test_split`, `OneClassSVM`, `roc_auc_score`, `.decision_function`, `.predict`.

<details><summary>Nápověda</summary>

Označení validační části se používá jen k **vyhodnocení** nastavení, nikdy k učení modelu. `decision_function` jednotřídní
SVM je uvnitř hranice kladné, jeho záporná hodnota je tedy skóre odchylky.

</details>

**Zamyslete se:** Jak daleko je jednotřídní model od SVM s učitelem z kroku 7? Kdy byste ho přesto použili?

**Vlastní kód:**

## 9. Charakteristika čerpadla: regrese s ε-trubicí a extrapolace (rozšiřující)

**Cíl / výstup:** tabulka `svr_results` s MAE tří modelů SVR uvnitř a vně učebního rozsahu, graf tří
křivek a tabulka `eps_results` s počtem podpůrných vektorů a MAE pro ε ∈ {0,05; 0,2; 0,5}.

1. Vezměte vývojové záznamy **v pořádku** (`y_dev == 0`). Vstup: jen `flow`, cíl: `pressure`.
   Učební rozsah: průtok < 65 m³/h; „nový rozsah“: průtok ≥ 65 (tam čerpadlo v učebních datech nikdy nepracovalo).
2. Naučte tři pipeline (škálování → `SVR`) s `C=10, epsilon=0.2`: lineární jádro, polynomiální `degree=2, coef0=1`, RBF `gamma=1`.
   Vypište MAE v učebním rozsahu a v novém rozsahu.
3. Vykreslete data a tři křivky pro průtok od 20 do 80.
4. U polynomiálního jádra měňte ε na 0,05; 0,2 a 0,5; vypište počet podpůrných vektorů a obě hodnoty MAE.

**Použijte:** `SVR`, `mean_absolute_error`, `.support_`, `plt.plot`.

<details><summary>Nápověda</summary>

`SVR` potřebuje dvourozměrný vstup: `data[["flow"]]`, ne `data["flow"]`. Šum tlaku kolem charakteristiky má směrodatnou
odchylku asi 0,45 bar, takže MAE kolem 0,27 je to nejlepší, čeho může uvnitř dat dosáhnout jakýkoli model.

</details>

**Zamyslete se:** Které jádro extrapoluje nejlépe a proč (snímek přednášky o extrapolaci)? Proč ε skoro nemění MAE,
ale výrazně mění počet podpůrných vektorů?

**Vlastní kód:**

## Shrnutí

* Třída, která obklopuje tu druhou (odchylka na kteroukoli stranu), potřebuje jádro; lineární model není lepší než náhoda.
* C a γ se ladí společně; nejlepší hodnota na kraji mřížky znamená mřížku rozšířit.
* Práh na skóre SVM se volí podle ceny chyb na vývojových datech, teprve potom se jednou otevře test.
* Jednotřídní SVM funguje bez označených poruch, za cenu slabšího výsledku.
* RBF jádro neextrapoluje; model se správným tvarem (zde parabola) ano.